<a href="https://colab.research.google.com/github/suphanatchanlek30/Super-AI-Engineer-Season-6-Math-VQA-Challenge/blob/main/Math_VQA_Challenge_600367_%E0%B8%A8%E0%B8%B8%E0%B8%A0%E0%B8%93%E0%B8%B1%E0%B8%90.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Thai Math VQA All-in-One Notebook

Notebook นี้รวมโค้ดทั้งหมดไว้ในไฟล์เดียวสำหรับโจทย์ `Super AI Engineer SS6 Individual Test: Math VQA Challenge`

สิ่งที่ทำได้:
- เช็ก dataset
- นิยาม helper functions และ classes ทั้งหมดใน notebook นี้
- รัน validation
- รัน submission
- ตรวจไฟล์ `submission.csv` ก่อนส่ง Kaggle


In [ ]:
# ถ้าเป็น environment ใหม่หรือ Colab ให้ติดตั้ง dependencies ก่อน
# !pip install -q numpy pandas pillow scikit-learn tqdm opencv-python rapidfuzz matplotlib easyocr pytesseract torch torchvision transformers accelerate timm

from __future__ import annotations

import json
import re
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from rapidfuzz import fuzz
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.model_selection import train_test_split
from tqdm import tqdm


In [ ]:
# Config
DATA_DIR = Path('./data')
OUTPUT_DIR = Path('./outputs')

OCR_ENGINE = 'easyocr'   # easyocr | tesseract | none
IMAGE_MODEL = 'resnet18' # resnet18 | none

DEVICE = 'auto'          # auto | cuda | cpu
OCR_GPU = 'auto'         # auto | true | false

VALID_SIZE = 0.2
RANDOM_STATE = 42
TOP_K = 5
OCR_WEIGHT = 0.55
IMAGE_WEIGHT = 0.45
USE_VLM = False
VLM_MODEL = 'Qwen/Qwen2.5-VL-3B-Instruct'
MAX_NEW_TOKENS = 64
BATCH_SIZE = 16


## Helpers and Pipeline Code

In [ ]:
THAI_DIGITS = str.maketrans('๐๑๒๓๔๕๖๗๘๙', '0123456789')

UNITS = [
    'ตารางเซนติเมตร', 'ตารางเมตร', 'ลูกบาศก์เซนติเมตร', 'ลูกบาศก์เมตร', 'ลูกบาศก์หน่วย',
    'ตารางวา', 'เซนติเมตร', 'มิลลิเมตร', 'กิโลเมตร', 'เมตร', 'องศา', 'หน่วย', 'จำนวน',
    'วิธี', 'แบบ', 'ค่า', 'ร้อยละ', 'ดอลลาร์', 'บาท', 'ปี', 'คน', 'ข้อ',
    'degrees', 'degree', 'squarecentimeters', 'squarecentimeter', 'yearsold'
]

LATEX_REPLACEMENTS = [
    (r'\\times', '*'),
    (r'\\cdot', '*'),
    (r'\\div', '/'),
    (r'\\pi', 'pi'),
    (r'\\pm', '+-'),
    (r'\\left', ''),
    (r'\\right', ''),
    (r'\\,', ''),
    (r'\\;', ''),
    (r'\\:', ''),
    (r'\\!', ''),
]

def _expand_frac(text: str) -> str:
    pattern = re.compile(r'\\frac\s*\{([^{}]+)\}\s*\{([^{}]+)\}')
    while True:
        new_text = pattern.sub(r'(\1)/(\2)', text)
        if new_text == text:
            return new_text
        text = new_text

def _expand_sqrt(text: str) -> str:
    pattern = re.compile(r'\\sqrt\s*\{([^{}]+)\}')
    while True:
        new_text = pattern.sub(r'sqrt(\1)', text)
        if new_text == text:
            return new_text
        text = new_text

def _expand_vectors(text: str) -> str:
    patterns = [
        re.compile(r'\\overrightarrow\s*\{([^{}]+)\}'),
        re.compile(r'\\overline\s*\{([^{}]+)\}'),
        re.compile(r'\\vec\s*\{([^{}]+)\}'),
    ]
    for pattern in patterns:
        text = pattern.sub(r'\1', text)
    return text

def _strip_units(text: str) -> str:
    for unit in sorted(UNITS, key=len, reverse=True):
        text = text.replace(unit, '')
    return text

def _drop_integer_parentheses(text: str) -> str:
    pattern = re.compile(r'\((-?\d+)\)')
    while True:
        new_text = pattern.sub(r'\1', text)
        if new_text == text:
            return new_text
        text = new_text

def _canonicalize_integer(text: str) -> str:
    stripped = text.strip()
    if not stripped:
        return stripped
    try:
        value = Decimal(stripped)
    except InvalidOperation:
        return stripped
    if value == value.to_integral_value():
        return str(int(value))
    normalized = format(value.normalize(), 'f')
    if '.' in normalized:
        normalized = normalized.rstrip('0').rstrip('.')
    return normalized

def normalize_answer(text: object) -> str:
    value = '' if text is None else str(text)
    value = value.strip().lower().translate(THAI_DIGITS)
    value = value.replace('$', '')
    value = _strip_units(value)
    value = _expand_frac(value)
    value = _expand_sqrt(value)
    value = _expand_vectors(value)
    for source, target in LATEX_REPLACEMENTS:
        value = re.sub(source, target, value)
    value = value.replace(' ', '')
    value = re.sub(r'[{}\\,]', '', value)
    value = _drop_integer_parentheses(value)
    return _canonicalize_integer(value)

def load_image(path: Path) -> Image.Image:
    return Image.open(path).convert('RGB')

def enhance_for_ocr(image: Image.Image) -> Image.Image:
    gray = ImageOps.grayscale(image)
    gray_np = np.array(gray)
    denoised = cv2.fastNlMeansDenoising(gray_np, None, 15, 7, 21)
    thresh = cv2.adaptiveThreshold(denoised, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, cv2.THRESH_BINARY, 35, 11)
    return Image.fromarray(thresh)

def resize_longest_side(image: Image.Image, longest_side: int = 1024) -> Image.Image:
    width, height = image.size
    scale = longest_side / max(width, height)
    if scale >= 1:
        return image
    new_size = (int(width * scale), int(height * scale))
    return image.resize(new_size, Image.Resampling.LANCZOS)

def simple_image_vector(image: Image.Image, size: int = 32) -> np.ndarray:
    gray = ImageOps.grayscale(image).resize((size, size), Image.Resampling.BILINEAR)
    arr = np.asarray(gray, dtype=np.float32) / 255.0
    vec = arr.reshape(-1)
    norm = np.linalg.norm(vec)
    return vec if norm == 0 else vec / norm

def resolve_image_root(data_dir: Path) -> Path:
    direct = data_dir / 'images'
    nested = data_dir / 'images' / 'images'
    if nested.exists() and nested.is_dir():
        return nested
    if direct.exists() and direct.is_dir():
        return direct
    raise FileNotFoundError(f'Could not find an images directory under: {data_dir}')

def resolve_image_path(data_dir: Path, image_path: str) -> Path:
    relative = Path(image_path)
    direct = data_dir / relative
    if direct.exists():
        return direct
    if relative.parts and relative.parts[0] == 'images':
        nested = data_dir / 'images' / relative
        if nested.exists():
            return nested
    nested_root = resolve_image_root(data_dir)
    fallback = nested_root / relative.name
    if fallback.exists():
        return fallback
    return direct

def build_tfidf(texts):
    cleaned = [text if str(text).strip() else '__empty__' for text in texts]
    vectorizer = TfidfVectorizer(analyzer='char_wb', ngram_range=(2, 5), min_df=1)
    matrix = vectorizer.fit_transform(cleaned)
    return vectorizer, matrix

@dataclass
class OCRResult:
    text: str
    engine: str

class OCRExtractor:
    def __init__(self, engine: str = 'easyocr', gpu: str = 'auto') -> None:
        self.engine = engine
        self.gpu = gpu
        self._reader = None
        if engine == 'easyocr':
            import easyocr
            import torch
            use_gpu = torch.cuda.is_available() if gpu == 'auto' else gpu == 'true'
            self._reader = easyocr.Reader(['th', 'en'], gpu=use_gpu)
        elif engine == 'tesseract':
            import pytesseract  # noqa: F401
        elif engine == 'none':
            pass
        else:
            raise ValueError(f'Unsupported OCR engine: {engine}')

    def extract(self, image_path: Path) -> OCRResult:
        if self.engine == 'none':
            return OCRResult(text='', engine=self.engine)
        image = enhance_for_ocr(load_image(image_path))
        if self.engine == 'easyocr':
            lines = self._reader.readtext(np.array(image), detail=0, paragraph=True)
            return OCRResult(text='\n'.join(lines), engine=self.engine)
        import pytesseract
        text = pytesseract.image_to_string(image, lang='tha+eng', config='--psm 6')
        return OCRResult(text=text, engine=self.engine)

class ImageFeatureExtractor:
    def __init__(self, model_name: str = 'resnet18', device: str = 'auto') -> None:
        self.model_name = model_name
        self.device = device
        self._backend = 'simple'
        self._model = None
        self._transform = None
        self._torch_device = None
        if model_name == 'none':
            return
        if model_name == 'resnet18':
            try:
                import torch
                from torchvision import models, transforms
                weights = models.ResNet18_Weights.DEFAULT
                model = models.resnet18(weights=weights)
                model.fc = torch.nn.Identity()
                torch_device = torch.device('cuda' if torch.cuda.is_available() else 'cpu') if device == 'auto' else torch.device(device)
                model = model.to(torch_device)
                model.eval()
                self._backend = 'torchvision'
                self._model = model
                self._torch_device = torch_device
                self._transform = transforms.Compose([
                    transforms.Resize((224, 224)),
                    transforms.ToTensor(),
                    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
                ])
            except Exception:
                self._backend = 'simple'

    def encode(self, image: Image.Image) -> np.ndarray:
        if self._backend == 'simple':
            return simple_image_vector(image)
        import torch
        tensor = self._transform(image).unsqueeze(0).to(self._torch_device)
        with torch.no_grad():
            features = self._model(tensor).squeeze(0).cpu().numpy().astype(np.float32)
        norm = np.linalg.norm(features)
        return features if norm == 0 else features / norm

@dataclass
class Candidate:
    answer: str
    score: float
    source_id: str

class RetrievalAnswerer:
    def __init__(self, train_df: pd.DataFrame, data_dir: Path, ocr_engine='easyocr', image_model='resnet18', ocr_weight=0.55, image_weight=0.45, batch_size=16, device='auto', ocr_gpu='auto'):
        self.train_df = train_df.reset_index(drop=True).copy()
        self.data_dir = data_dir
        self.ocr_weight = ocr_weight
        self.image_weight = image_weight
        self.batch_size = batch_size
        self.ocr = OCRExtractor(ocr_engine, gpu=ocr_gpu)
        self.image_extractor = ImageFeatureExtractor(image_model, device=device)
        self.train_df['ocr_text'] = self._extract_ocr_for_frame(self.train_df)
        self.vectorizer, self.train_ocr_matrix = build_tfidf(self.train_df['ocr_text'].fillna(''))
        self.train_image_matrix = self._build_image_matrix(self.train_df)

    def _extract_ocr_for_frame(self, frame: pd.DataFrame):
        texts = []
        for _, row in tqdm(frame.iterrows(), total=len(frame), desc='OCR(train)'):
            image_path = resolve_image_path(self.data_dir, row['image_path'])
            try:
                texts.append(self.ocr.extract(image_path).text)
            except Exception:
                texts.append('')
        return texts

    def _build_image_matrix(self, frame: pd.DataFrame) -> np.ndarray:
        features = []
        for _, row in tqdm(frame.iterrows(), total=len(frame), desc='IMG(train)'):
            image_path = resolve_image_path(self.data_dir, row['image_path'])
            image = resize_longest_side(load_image(image_path), longest_side=1024)
            features.append(self.image_extractor.encode(image))
        return np.vstack(features)

    def _query_ocr(self, image_path: Path) -> str:
        try:
            return self.ocr.extract(image_path).text
        except Exception:
            return ''

    def _query_image_vector(self, image_path: Path) -> np.ndarray:
        image = resize_longest_side(load_image(image_path), longest_side=1024)
        return self.image_extractor.encode(image)

    def retrieve(self, image_path: Path, top_k: int = 5):
        query_text = self._query_ocr(image_path)
        query_text_vector = self.vectorizer.transform([query_text if query_text.strip() else '__empty__'])
        ocr_scores = cosine_similarity(query_text_vector, self.train_ocr_matrix).ravel()
        image_vector = self._query_image_vector(image_path)[None, :]
        image_scores = cosine_similarity(image_vector, self.train_image_matrix).ravel()
        combined = (self.ocr_weight * ocr_scores) + (self.image_weight * image_scores)
        best_idx = np.argsort(combined)[::-1][:top_k]
        candidates = []
        for idx in best_idx:
            train_row = self.train_df.iloc[idx]
            lexical_boost = fuzz.partial_ratio(query_text, train_row['ocr_text']) / 100.0 if query_text else 0.0
            score = float(combined[idx] + 0.1 * lexical_boost)
            candidates.append(Candidate(answer=str(train_row['answer']), score=score, source_id=str(train_row['id'])))
        return candidates

    @staticmethod
    def vote(candidates):
        by_norm = {}
        for candidate in candidates:
            normalized = normalize_answer(candidate.answer)
            if normalized not in by_norm or candidate.score > by_norm[normalized][0]:
                by_norm[normalized] = (candidate.score, candidate.answer)
        winner = max(by_norm.values(), key=lambda item: item[0])
        return winner[1]

PROMPT = '''You are solving a Thai mathematics visual question.
Read the image carefully and solve the problem.
Return only the final answer.
Rules:
- No explanation
- Use Arabic digits
- Fraction as a/b
- Square root as sqrt(...)
- Keep the answer short
Final answer:
'''

@dataclass
class VLMResult:
    answer: str
    model_name: str

class VLMAnswerer:
    def __init__(self, model_name: str, device: str = 'auto', max_new_tokens: int = 64) -> None:
        import torch
        from transformers import AutoProcessor, AutoModelForImageTextToText
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.torch = torch
        self.processor = AutoProcessor.from_pretrained(model_name, trust_remote_code=True)
        self.model = AutoModelForImageTextToText.from_pretrained(
            model_name,
            torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
            device_map=device,
            trust_remote_code=True,
        )

    def answer(self, image_path: Path) -> VLMResult:
        image = Image.open(image_path).convert('RGB')
        messages = [{'role': 'user', 'content': [{'type': 'image', 'image': image}, {'type': 'text', 'text': PROMPT}]}]
        text = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
        inputs = self.processor(text=[text], images=[image], return_tensors='pt')
        inputs = {k: v.to(self.model.device) if hasattr(v, 'to') else v for k, v in inputs.items()}
        output = self.model.generate(**inputs, max_new_tokens=self.max_new_tokens)
        decoded = self.processor.batch_decode(output, skip_special_tokens=True)[0]
        answer = decoded.split('Final answer:')[-1].strip().splitlines()[-1].strip()
        return VLMResult(answer=answer, model_name=self.model_name)

@dataclass
class PipelineConfig:
    data_dir: Path
    output_dir: Path
    mode: str = 'both'
    ocr_engine: str = 'easyocr'
    image_model: str = 'resnet18'
    valid_size: float = 0.2
    random_state: int = 42
    top_k: int = 5
    ocr_weight: float = 0.55
    image_weight: float = 0.45
    use_vlm: bool = False
    vlm_model: str = 'Qwen/Qwen2.5-VL-3B-Instruct'
    device: str = 'auto'
    ocr_gpu: str = 'auto'
    max_new_tokens: int = 64
    batch_size: int = 16

def normalized_accuracy(truth: pd.Series, pred: pd.Series) -> float:
    truth_norm = truth.map(normalize_answer)
    pred_norm = pred.map(normalize_answer)
    return float((truth_norm == pred_norm).mean())

def _candidate_data_dirs(root: Path):
    candidates = [root]
    if root.exists() and root.is_dir():
        for child in root.iterdir():
            if child.is_dir():
                candidates.append(child)
    return candidates

def _resolve_data_dir(data_dir: Path) -> Path:
    if not data_dir.exists():
        raise FileNotFoundError(f'Dataset directory does not exist: {data_dir}')
    if not data_dir.is_dir():
        raise NotADirectoryError(f'Dataset path is not a directory: {data_dir}')
    required = ('train.csv', 'test.csv')
    if all((data_dir / name).exists() for name in required):
        return data_dir
    for candidate in _candidate_data_dirs(data_dir):
        if all((candidate / name).exists() for name in required):
            return candidate
    raise FileNotFoundError(f'Could not find competition CSV files under: {data_dir}')

def _load_frames(data_dir: Path):
    resolved_dir = _resolve_data_dir(data_dir)
    train = pd.read_csv(resolved_dir / 'train.csv')
    test = pd.read_csv(resolved_dir / 'test.csv')
    images_dir = resolve_image_root(resolved_dir)
    image_count = sum(1 for _ in images_dir.glob('*.jpg'))
    if image_count == 0:
        raise FileNotFoundError(f'No JPG files found in: {images_dir}')
    train['id'] = train['id'].astype(str)
    test['id'] = test['id'].astype(str)
    train.attrs['resolved_data_dir'] = str(resolved_dir)
    test.attrs['resolved_data_dir'] = str(resolved_dir)
    return train, test

def _predict_frame(frame: pd.DataFrame, retriever: RetrievalAnswerer, config: PipelineConfig) -> pd.DataFrame:
    vlm = VLMAnswerer(config.vlm_model, device=config.device, max_new_tokens=config.max_new_tokens) if config.use_vlm else None
    rows = []
    for _, row in tqdm(frame.iterrows(), total=len(frame), desc='Predict'):
        image_path = resolve_image_path(retriever.data_dir, row['image_path'])
        candidates = retriever.retrieve(image_path, top_k=config.top_k)
        if vlm is not None:
            try:
                vlm_result = vlm.answer(image_path)
                candidates.append(Candidate(answer=vlm_result.answer, score=10.0, source_id='vlm'))
            except Exception:
                pass
        prediction = retriever.vote(candidates)
        record = {
            'id': row['id'],
            'image_path': row['image_path'],
            'prediction': prediction,
            'prediction_normalized': normalize_answer(prediction),
            'candidate_answers': ' | '.join(f'{c.answer}<{c.source_id}:{c.score:.3f}>' for c in candidates),
        }
        if 'answer' in row:
            record['answer'] = row['answer']
            record['answer_normalized'] = normalize_answer(row['answer'])
        rows.append(record)
    return pd.DataFrame(rows)

def run_pipeline(config: PipelineConfig) -> None:
    config.output_dir.mkdir(parents=True, exist_ok=True)
    train_df, test_df = _load_frames(config.data_dir)
    resolved_data_dir = Path(train_df.attrs['resolved_data_dir'])
    metrics = {}
    if config.mode in {'validate', 'both'}:
        train_part, valid_part = train_test_split(train_df, test_size=config.valid_size, random_state=config.random_state, shuffle=True)
        retriever = RetrievalAnswerer(train_df=train_part, data_dir=resolved_data_dir, ocr_engine=config.ocr_engine, image_model=config.image_model, ocr_weight=config.ocr_weight, image_weight=config.image_weight, batch_size=config.batch_size, device=config.device, ocr_gpu=config.ocr_gpu)
        valid_predictions = _predict_frame(valid_part.reset_index(drop=True), retriever, config)
        score = normalized_accuracy(valid_predictions['answer'], valid_predictions['prediction'])
        valid_predictions.to_csv(config.output_dir / 'validation_predictions.csv', index=False)
        metrics['validation_accuracy'] = score
        metrics['validation_rows'] = len(valid_predictions)
    if config.mode in {'submit', 'both'}:
        retriever = RetrievalAnswerer(train_df=train_df, data_dir=resolved_data_dir, ocr_engine=config.ocr_engine, image_model=config.image_model, ocr_weight=config.ocr_weight, image_weight=config.image_weight, batch_size=config.batch_size, device=config.device, ocr_gpu=config.ocr_gpu)
        test_predictions = _predict_frame(test_df.reset_index(drop=True), retriever, config)
        submission = test_predictions[['id', 'prediction']].rename(columns={'prediction': 'answer'})
        submission.to_csv(config.output_dir / 'submission.csv', index=False)
        test_predictions.to_csv(config.output_dir / 'test_predictions_debug.csv', index=False)
        metrics['submission_rows'] = len(submission)
    metrics['mode'] = config.mode
    metrics['resolved_data_dir'] = str(resolved_data_dir)
    with open(config.output_dir / 'metrics.json', 'w', encoding='utf-8') as fp:
        json.dump(metrics, fp, ensure_ascii=False, indent=2)


## Dataset Check

In [ ]:
train_csv = DATA_DIR / 'train.csv'
test_csv = DATA_DIR / 'test.csv'
sample_csv = DATA_DIR / 'sample_submission.csv'
assert train_csv.exists(), f'missing: {train_csv}'
assert test_csv.exists(), f'missing: {test_csv}'
assert sample_csv.exists(), f'missing: {sample_csv}'

image_root = resolve_image_root(DATA_DIR)
jpg_count = len(list(image_root.glob('*.jpg')))
train_df = pd.read_csv(train_csv)
test_df = pd.read_csv(test_csv)

print('DATA_DIR      :', DATA_DIR.resolve())
print('IMAGE_ROOT    :', image_root.resolve())
print('train rows    :', len(train_df))
print('test rows     :', len(test_df))
print('jpg files     :', jpg_count)
display(train_df.head())


## Run Validation

In [ ]:
validate_config = PipelineConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    mode='validate',
    ocr_engine=OCR_ENGINE,
    image_model=IMAGE_MODEL,
    valid_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    top_k=TOP_K,
    ocr_weight=OCR_WEIGHT,
    image_weight=IMAGE_WEIGHT,
    use_vlm=USE_VLM,
    vlm_model=VLM_MODEL,
    device=DEVICE,
    ocr_gpu=OCR_GPU,
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=BATCH_SIZE,
)
run_pipeline(validate_config)
print('validation done')


In [ ]:
metrics = json.loads((OUTPUT_DIR / 'metrics.json').read_text(encoding='utf-8'))
print(json.dumps(metrics, ensure_ascii=False, indent=2))
validation_df = pd.read_csv(OUTPUT_DIR / 'validation_predictions.csv')
display(validation_df.head(10))


## Run Submission

In [ ]:
submit_config = PipelineConfig(
    data_dir=DATA_DIR,
    output_dir=OUTPUT_DIR,
    mode='submit',
    ocr_engine=OCR_ENGINE,
    image_model=IMAGE_MODEL,
    valid_size=VALID_SIZE,
    random_state=RANDOM_STATE,
    top_k=TOP_K,
    ocr_weight=OCR_WEIGHT,
    image_weight=IMAGE_WEIGHT,
    use_vlm=USE_VLM,
    vlm_model=VLM_MODEL,
    device=DEVICE,
    ocr_gpu=OCR_GPU,
    max_new_tokens=MAX_NEW_TOKENS,
    batch_size=BATCH_SIZE,
)
run_pipeline(submit_config)
print('submission done')


In [ ]:
submission_path = OUTPUT_DIR / 'submission.csv'
debug_path = OUTPUT_DIR / 'test_predictions_debug.csv'
submission_df = pd.read_csv(submission_path)
debug_df = pd.read_csv(debug_path)

print('submission path:', submission_path.resolve())
print('rows           :', len(submission_df))
print('columns        :', list(submission_df.columns))
print('empty answers  :', submission_df['answer'].isna().sum())

assert list(submission_df.columns) == ['id', 'answer']
assert len(submission_df) == len(test_df)
display(submission_df.head(20))


## Ready To Upload

ไฟล์พร้อมส่งคือ `outputs/submission.csv`
